# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mansi-cs/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!git clone https://github.com/mansi-cs/flyrank-ml-internship.git
%cd flyrank-ml-internship
!python scripts/01_prepare_features.py
!python scripts/ml_utils.py


Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 177, done.
remote: Counting objects: 100% (177/177), done.
remote: Compressing objects: 100% (133/133), done.
remote: Total 177 (delta 80), reused 92 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (177/177), 1.87 MiB | 1.59 MiB/s, done.
Resolving deltas: 100% (80/80), done.
/content/flyrank-ml-internship
Prepared 30,000 rows from 30,000 raw rows
Wrote /content/flyrank-ml-internship/data/processed/refresh_feature_vector.csv


In [4]:

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from scripts.ml_utils import (
    MODEL_NUMERIC_FEATURES,
    MODEL_CATEGORICAL_FEATURES
)

In [6]:
import pandas as pd
df = pd.read_csv("data/processed/refresh_feature_vector.csv")

print(df.shape)
print(df["is_declining_label"].value_counts())

(30000, 52)
is_declining_label
1    16262
0    13738
Name: count, dtype: int64


In [15]:
stale = (
    df["days_since_last_update"] >= 180
).astype(int)

visible = (
    df["impressions_90d"] >= 500
).astype(int)

low_ctr = (
    df["ctr"] < df["ctr"].median()
).astype(int)

df["baseline_action_score"] = (
    stale * 2
    + visible * 2
    + low_ctr
)
print("baseline_action_score" in df.columns)

True


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

My Week 4 baseline used a transparent rule based on staleness, visibility, and CTR. For Week 5 I selected Logistic Regression because it produces probabilities that can be used for ranking refresh opportunities and is easy to interpret. I chose a simple model first so that any improvement over the baseline can be clearly explained rather than relying on model complexity.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [16]:
groups = df["client_id"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(df, groups=groups)
)

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

print("Train:", train_df.shape)
print("Test:", test_df.shape)

Train: (23837, 53)
Test: (6163, 53)


A grouped split by client_id was used so that pages from the same client do not appear in both train and test sets. This provides a more honest estimate of performance on unseen clients.

In [17]:
X_train = train_df[
    MODEL_NUMERIC_FEATURES +
    MODEL_CATEGORICAL_FEATURES
]

X_test = test_df[
    MODEL_NUMERIC_FEATURES +
    MODEL_CATEGORICAL_FEATURES
]

y_train = train_df["is_declining_label"]
y_test = test_df["is_declining_label"]

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [20]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median"))
            ]),
            MODEL_NUMERIC_FEATURES
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore"))
            ]),
            MODEL_CATEGORICAL_FEATURES
        )
    ]
)

In [21]:
from sklearn.linear_model import LogisticRegression

model = Pipeline([
    ("prep", preprocessor),
    (
        "lr",
        LogisticRegression(
            max_iter=1000
        )
    )
])

model.fit(X_train, y_train)

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['search_volume',
                                                   'competition', 'cpc',
                                                   'word_count', 'char_count',
                                                   'log_impressions_90d',
                                                   'log_clicks_90d',
                                                   'log_sessions_90d',
                                                   'log_ai_sessions_90d',
                                                   'days_with_impressions',
                                                   'days_with_sessions',
                                                   'content_age_days',
                                                   'days_since_last_...
                                                   'engagement_rate',
                                                   'scroll_rate',
                                                   'ai_traffic_pct']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['competition_level',
                                                   'content_type',
                                                   'main_intent', 'age_tier',
                                                   'freshness_tier',
                                                   'word_count_tier',
                                                   'impression_tier',
                                                   'position_tier'])])),
                ('lr', LogisticRegression(max_iter=1000))])

In [22]:
model_scores = model.predict_proba(X_test)[:, 1]

In [23]:
import numpy as np

def precision_at_k(y_true, scores, k):
    top_idx = np.argsort(scores)[::-1][:k]
    return y_true.iloc[top_idx].mean()

In [24]:
p20_model = precision_at_k(
    y_test,
    model_scores,
    20
)

p50_model = precision_at_k(
    y_test,
    model_scores,
    50
)

print("Model Precision@20:", round(p20_model, 3))
print("Model Precision@50:", round(p50_model, 3))

Model Precision@20: 0.75
Model Precision@50: 0.74


In [25]:
baseline_scores = test_df[
    "baseline_action_score"
]

In [26]:
p20_baseline = precision_at_k(
    y_test,
    baseline_scores,
    20
)

p50_baseline = precision_at_k(
    y_test,
    baseline_scores,
    50
)

print(
    "Baseline Precision@20:",
    round(p20_baseline, 3)
)

print(
    "Baseline Precision@50:",
    round(p50_baseline, 3)
)

Baseline Precision@20: 0.7
Baseline Precision@50: 0.66


In [27]:
comparison = pd.DataFrame({
    "Method": [
        "Week 4 Baseline",
        "Logistic Regression"
    ],
    "Precision@20": [
        p20_baseline,
        p20_model
    ],
    "Precision@50": [
        p50_baseline,
        p50_model
    ]
})

comparison

,Method,Precision@20,Precision@50
0,Week 4 Baseline,0.70,0.66
1,Logistic Regression,0.75,0.74


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [28]:
predictions = model.predict(X_test)
results = test_df.copy()

results["actual"] = y_test.values
results["predicted"] = predictions
results["probability"] = model_scores

results[
    [
        "content_id",
        "actual",
        "predicted",
        "probability",
        "ctr",
        "impressions_90d",
        "days_since_last_update"
    ]
].head()

,content_id,actual,predicted,probability,ctr,impressions_90d,days_since_last_update
0,content_304f48230142,1,1,0.707938,0.76,3803,20
1,content_a1fb4e703a9e,1,1,0.593434,0.05,15320,25
5,content_d4084a4bc775,1,1,0.820442,0.03,3970,20
13,content_a5a2fbc76336,0,1,0.659381,0.00,307,103
19,content_af865035b328,1,1,0.650051,2.02,99,20


In [29]:
false_positives = results[
    (results["actual"] == 0) &
    (results["predicted"] == 1)
]

print("False positives:", len(false_positives))

false_positives[
    [
        "content_id",
        "probability",
        "ctr",
        "impressions_90d",
        "days_since_last_update"
    ]
].head(5)

False positives: 1539


,content_id,probability,ctr,impressions_90d,days_since_last_update
13,content_a5a2fbc76336,0.659381,0.00,307,103
26,content_72c5c2d73e5a,0.654283,0.12,2426,13
36,content_bce275871a25,0.703566,1.35,371,20
56,content_dcebfd222b10,0.567054,0.00,16,20
64,content_685de0e3b7cb,0.794950,0.11,2639,8


In [30]:
false_negatives = results[
    (results["actual"] == 1) &
    (results["predicted"] == 0)
]

print("False negatives:", len(false_negatives))

false_negatives[
    [
        "content_id",
        "probability",
        "ctr",
        "impressions_90d",
        "days_since_last_update"
    ]
].head(5)

False negatives: 1078


,content_id,probability,ctr,impressions_90d,days_since_last_update
23,content_2da6ae9d0882,0.314427,0.34,297,20
39,content_4595e8704e07,0.308722,0.00,4,104
47,content_40cb4af260c0,0.453434,0.00,8,20
54,content_ff8ea1364b59,0.439005,0.00,170,22
58,content_caff51984338,0.361353,0.00,71,20


In [ ]:
feature_names = model.named_steps[
    "prep"
].get_feature_names_out()

coefficients = model.named_steps[
    "lr"
].coef_[0]

importance = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients
})

importance["abs_coefficient"] = (
    importance["coefficient"].abs()
)

importance = importance.sort_values(
    "abs_coefficient",
    ascending=False
)

importance.head(15)

In [31]:
feature_names = model.named_steps[
    "prep"
].get_feature_names_out()

coefficients = model.named_steps[
    "lr"
].coef_[0]

importance = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients
})

importance["abs_coefficient"] = (
    importance["coefficient"].abs()
)

importance = importance.sort_values(
    "abs_coefficient",
    ascending=False
)

importance.head(15)

,feature,coefficient,abs_coefficient
5,num__log_impressions_90d,0.084176,0.084176
13,num__ctr,-0.070539,0.070539
7,num__log_sessions_90d,-0.044402,0.044402
51,cat__position_tier_top_3,-0.038587,0.038587
6,num__log_clicks_90d,-0.031497,0.031497
21,cat__competition_level_unknown,-0.023065,0.023065
23,cat__content_type_feedly article,-0.023045,0.023045
29,cat__main_intent_unknown,-0.022566,0.022566
38,cat__word_count_tier_1000-2000,0.022062,0.022062
46,cat__impression_tier_moderate,-0.021650,0.021650


### Overall interpretation:
Logistic Regression improved the Week 4 baseline on both ranking metrics, increasing Precision@20 from 0.70 to 0.75 and Precision@50 from 0.66 to 0.74. The error analysis shows that the model still produces both false positives and false negatives, particularly when pages have mixed visibility, CTR, and freshness signals. The coefficient analysis indicates that the model mainly relies on traffic, engagement, and search-position features. Overall, the results suggest that the model provides a stronger ranking signal than the hand-written baseline while remaining relatively simple and interpretable.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.